# Social Network Analysis Of Users and Recipes from Food.com

### By: Lars Hidding, Jasper Trooster and Julian van Rijckevorsel

#### Dataset

Food.com

https://www.kaggle.com/datasets/shuyangli94/food-com-recipes-and-user-interactions

#### Contents
1. Preparation (loading, cleaning, building the graph)
2. Basic metrics (nodes, edges, user roles, density)
3. Degree distributions
4. Centralities (contributors, reviewers, structural)
5. Communities (cliques, homophily, bridges, partitioning)
6. Ingredient and tag profiles

#### AI usage
I used AI to reorder and annotate my analysis, aswell as write some of the functionality, mainly the visualization

#### Note
As apparent in the report I was not able to run all of the analysis for the cliques, communities and left out the bridging among other things, this is mainly due to the fact that the dataset is fairly large and my 16 GB was used up completely, my computer has crashed several times trying to run this...

## 1. Preparation

### 1.1 Importing and loading the data

In [ ]:
import os
import ast
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse
from scipy.sparse.linalg import eigsh
from scipy.sparse.csgraph import shortest_path
from sklearn.metrics import normalized_mutual_info_score as nmi

os.makedirs("figures/metrics", exist_ok=True)
rng = np.random.default_rng(0)

recipes = pd.read_csv("data/RAW_recipes.csv", usecols=["name", "id", "contributor_id", "submitted", "tags", "ingredients"])
reviews = pd.read_csv("data/RAW_interactions.csv", usecols=["user_id", "recipe_id", "date", "rating"])

# The dataset puts the tags and ingredients as an array in a string. E.g. "['vegan', 'kosher']", so we turn it back into an array
recipes["tags"] = recipes["tags"].apply(ast.literal_eval)
recipes["ingredients"] = recipes["ingredients"].apply(ast.literal_eval)

# Now we make each tag-recipe and ingredient-recipe pairing its own row
recipe_tag = recipes[["id", "tags"]].explode("tags").dropna().drop_duplicates()
recipe_ing = recipes[["id", "ingredients"]].explode("ingredients").dropna().drop_duplicates()

### 1.2 Intermezzo 0-star ratings

While researching why there were so many self-reviews with zero stars, we discovered that these were comments or replies to other reviews,
and did not have any actual star-rating on the website. We later discovered that this also held true for reviews by other users.
E.g. https://www.food.com/recipe/dolmades-with-yogurt-mint-sauce-316. Therefore we remove all reviews with zero stars.

In [ ]:
# Self-reviews
contributor_by_recipe = recipes.set_index("id")["contributor_id"]
self_reviews = reviews[reviews["user_id"] == reviews["recipe_id"].map(contributor_by_recipe)]
print(self_reviews.groupby("rating")["rating"].count())

# Number of zero-star reviews (for the report), counted before removing them
n_zero = (reviews["rating"] == 0).sum()
print(f"zero-star reviews: {n_zero} ({n_zero / len(reviews):.2%} of all reviews)")

# Here we remove the reviews with rating 0
reviews = reviews[reviews["rating"] > 0]

### 1.3 Building the graph

In [ ]:
# We use a multi directed graph, since a contributor can technically review their own recipes
G = nx.MultiDiGraph()

# Adding the recipe and tag nodes
G.add_nodes_from(
    (f"r{r.id}", {"kind": "recipe", "name": r.name})
    for r in recipes.itertuples()
)
G.add_nodes_from((f"t{t}", {"kind": "tag", "label": t}) for t in recipe_tag["tags"].unique())

# Adding the user nodes for both contributors as well as reviewers, because they can be both
users = pd.concat([recipes["contributor_id"], reviews["user_id"]]).unique()
G.add_nodes_from((f"u{u}", {"kind": "user"}) for u in users)

# Adding all of the edges
G.add_edges_from(
    (f"u{r.contributor_id}", f"r{r.id}", "contributed", {"type": "contributed"})
    for r in recipes.itertuples()
)
G.add_edges_from(
    (f"u{r.user_id}", f"r{r.recipe_id}", "reviewed",
     {"type": "reviewed", "rating": r.rating, "date": r.date})
    for r in reviews.itertuples()
)
G.add_edges_from(
    (f"r{r.id}", f"t{r.tags}", "tagged", {"type": "tagged"})
    for r in recipe_tag.itertuples()
)

print(G)

## 2. Basic metrics (#nodes, #links, user roles, density)

In [ ]:
S = nx.DiGraph(G)
U = S.to_undirected()
kind = nx.get_node_attributes(S, "kind")

# Per-kind
print(pd.Series(kind).value_counts())
print(pd.Series([k for _, _, k in G.edges(keys=True)]).value_counts())

### 2.1 Contributors versus reviewers

In [ ]:
contributors = set(recipes["contributor_id"].unique())
reviewers    = set(reviews["user_id"].unique())

all_users         = contributors | reviewers
both              = contributors & reviewers
only_contributors = contributors - reviewers
only_reviewers    = reviewers - contributors

summary = pd.Series({
    "Total users":        len(all_users),
    "Total contributors": len(contributors),
    "only contributors":  len(only_contributors),
    "Total reviewers":    len(reviewers),
    "only reviewers":     len(only_reviewers),
    "both":               len(both),
})
summary_pct = (summary / len(all_users) * 100).round(2)
print(pd.DataFrame({"count": summary, "% of users": summary_pct}))

### 2.2 Density

In [ ]:
print("Density (directed):", nx.density(S))
print("Density (undirected):", nx.density(U))

# Density per layer: edges / (nodes on side A * nodes on side B), over the participating nodes
n_recipes = recipes["id"].nunique()
layers = pd.DataFrame({
    "edges":  [len(recipes), len(reviews), len(recipe_tag)],
    "side A": [len(contributors), len(reviewers), n_recipes],
    "side B": [n_recipes, n_recipes, recipe_tag["tags"].nunique()],
}, index=["contribution (user-recipe)", "review (user-recipe)", "tagging (recipe-tag)"])
layers["density"] = layers["edges"] / (layers["side A"] * layers["side B"])
print(layers.to_string(float_format=lambda x: f"{x:.3e}"))

## 3. Degree distributions

In [ ]:
# Per-layer degrees
reviews_per_user   = reviews.groupby("user_id").size()
contribs_per_user  = recipes.groupby("contributor_id").size()
reviews_per_recipe = reviews.groupby("recipe_id").size().reindex(recipes["id"], fill_value=0)
tags_per_recipe    = recipe_tag.groupby("id").size()
recipes_per_tag    = recipe_tag.groupby("tags").size()

n_unreviewed = (reviews_per_recipe == 0).sum()
print(f"recipes without reviews: {n_unreviewed} ({n_unreviewed / len(reviews_per_recipe):.2%})")

panels = [
    ("User out-degree: reviews written",     reviews_per_user,   "#4c78a8"),
    ("User out-degree: recipes contributed", contribs_per_user,  "#4c78a8"),
    ("Recipe in-degree: reviews received",   reviews_per_recipe, "#e4572e"),
    ("Recipe out-degree: tags per recipe",   tags_per_recipe,    "#e4572e"),
    ("Tag in-degree: recipes per tag",       recipes_per_tag,    "#59a14f"),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (title, s, color) in zip(axes.flat, panels):
    s = s[s > 0]

    if title == "Recipe out-degree: tags per recipe":
        ax.hist(s, bins=range(1, s.max() + 2), color=color, edgecolor="white")
    else:
        bins = np.logspace(0, np.log10(s.max() + 1), 40)
        ax.hist(s, bins=bins, color=color, edgecolor="white")
        ax.set_xscale("log")

    ax.set_yscale("log")
    ax.set_title(title)
    ax.set_xlabel("degree")
    ax.set_ylabel("number of nodes")

axes.flat[-1].axis("off")
plt.tight_layout()
plt.savefig("figures/metrics/degree_distribution.png", dpi=200)
plt.show()

## 4. Centralities

### 4.1 Contributors: reviews received

In [ ]:
reviews_per_contributor = (
    recipes.set_index("id")["contributor_id"]
    .to_frame()
    .assign(n_reviews=reviews_per_recipe)
    .groupby("contributor_id")["n_reviews"]
    .sum()
    .astype(int)
)

cc = pd.DataFrame({
    "total_reviews": reviews_per_contributor,
    "n_recipes": contribs_per_user,
}).fillna(0)
cc["reviews_per_recipe"] = cc["total_reviews"] / cc["n_recipes"]

print(cc.nlargest(10, "total_reviews"))
print(cc.query("n_recipes >= 5").nlargest(10, "reviews_per_recipe"))
print("Spearman:", cc[["total_reviews", "reviews_per_recipe"]].corr("spearman").iloc[0, 1])

In [ ]:
plot_cc = cc[cc["total_reviews"] > 0]
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(plot_cc["total_reviews"], plot_cc["reviews_per_recipe"], s=6, alpha=.3, color="#4c78a8")
ax.set_xscale("log"); ax.set_yscale("log")
label_ids = list(plot_cc.nlargest(3, "total_reviews").index) + list(plot_cc[plot_cc["n_recipes"] >= 5].nlargest(3, "reviews_per_recipe").index)
for u in label_ids:
    ax.annotate(str(u), (plot_cc.loc[u, "total_reviews"], plot_cc.loc[u, "reviews_per_recipe"]),
                fontsize=8, xytext=(4, 3), textcoords="offset points")
ax.set_xlabel("total reviews received"); ax.set_ylabel("mean reviews per recipe")
plt.tight_layout()
plt.savefig("figures/metrics/contributor_ranking.png", dpi=200)
plt.show()

### 4.2 Reviewers: agreement with other ratings

In [ ]:
g = reviews.groupby("recipe_id")["rating"].agg(["sum", "count"])
r = reviews.join(g, on="recipe_id").query("count > 1").copy()
r["err"] = (r["rating"] - (r["sum"] - r["rating"]) / (r["count"] - 1)).abs()
agree = r.groupby("user_id")["err"].agg(n="size", mae="mean").query("n >= 20")
print(agree.describe())

### 4.3 Contributor-reviewer graph

In [ ]:
rev = reviews.assign(contributor_id=reviews["recipe_id"].map(contributor_by_recipe))
rev = rev.dropna(subset=["contributor_id"])
rev["contributor_id"] = rev["contributor_id"].astype(reviews["user_id"].dtype)
rev = rev[rev["user_id"] != rev["contributor_id"]]            # no self-reviews

edges = rev.groupby(["user_id", "contributor_id"]).size().reset_index(name="weight")
D = nx.from_pandas_edgelist(edges, "user_id", "contributor_id", "weight", create_using=nx.DiGraph)

# undirected version, summing weights of reciprocal pairs
e = edges.assign(a=edges[["user_id", "contributor_id"]].min(axis=1),
                 b=edges[["user_id", "contributor_id"]].max(axis=1))
UG = nx.from_pandas_edgelist(e.groupby(["a", "b"], as_index=False)["weight"].sum(), "a", "b", "weight")
print(UG.number_of_nodes(), UG.number_of_edges(), nx.number_connected_components(UG))

### 4.4 Structural centralities (eigenvector, Katz, closeness, betweenness)

In [ ]:
L = UG.subgraph(max(nx.connected_components(UG), key=len))
nodes = list(L)
A = nx.to_scipy_sparse_array(L, nodelist=nodes, weight=None, format="csr", dtype=float)
print("LCC share of nodes:", len(nodes) / UG.number_of_nodes())

# Eigenvector (exact, sparse eigensolver)
lam, vec = eigsh(A, k=1, which="LA")
print("lambda_max:", lam[0])
eig = pd.Series(np.abs(vec[:, 0]), index=nodes)

# Katz (alpha just below 1 / lambda_max, beta = 1)
alpha, x = 0.85 / lam[0], np.ones(len(nodes))
for _ in range(200):
    xn = alpha * (A @ x) + 1
    if np.abs(xn - x).sum() < 1e-8:
        break
    x = xn
katz = pd.Series(xn, index=nodes)

piv = rng.choice(len(nodes), 500, replace=False)
ds = sum(shortest_path(A, method="D", unweighted=True, indices=b).sum(0) for b in np.array_split(piv, 25))
clo = pd.Series(500 / ds, index=nodes)

bet = pd.Series(nx.betweenness_centrality(L, k=500, seed=0))

cent = pd.DataFrame({"eigenvector": eig, "katz": katz, "closeness": clo, "betweenness": bet})
print(cent.corr("spearman").round(2))
print(cent.apply(lambda c: c.nlargest(5).index.tolist()))

In [ ]:
d = pd.Series(dict(L.degree()))
print(pd.concat([cent, d.rename("degree")], axis=1).corr("spearman")["degree"].round(2))
print(cent.rank(ascending=False).loc[[89831, 37449, 4470]])

## 5. Communities

### 5.1 Cliques

In [ ]:
H = UG.subgraph(both).copy()
sizes = pd.Series([len(c) for c in nx.find_cliques(H)]).value_counts().sort_index()
print("H nodes/edges:", H.number_of_nodes(), H.number_of_edges())
print(sizes)
print("clique number:", sizes.index.max())
print("reciprocity:", nx.reciprocity(D))                 # share of edges that are mutual
print("avg clustering (both-users):", nx.average_clustering(H))

### 5.2 Homophily

In [ ]:
nx.set_node_attributes(UG, {n: (n in contributors) for n in UG}, "contrib")
print("share contributors:", np.mean([n in contributors for n in UG]))
print("contributor assortativity:", nx.attribute_assortativity_coefficient(UG, "contrib"))
print("degree assortativity:", nx.degree_assortativity_coefficient(UG))

### 5.3 Bridges

In [ ]:
deg = dict(UG.degree())
br = list(nx.bridges(UG))
pend = sum(deg[u] == 1 or deg[v] == 1 for u, v in br)
print("bridges:", len(br), "pendant:", pend, "non-pendant:", len(br) - pend)

### 5.4 Partitioning algorithm

In [ ]:
# Louvain on the weighted contributor-reviewer graph, two seeds to check stability
p0 = nx.community.louvain_communities(UG, weight="weight", seed=0)
p1 = nx.community.louvain_communities(UG, weight="weight", seed=1)
print("communities:", len(p0), "modularity:", nx.community.modularity(UG, p0, weight="weight"))
print("top sizes:", sorted(map(len, p0), reverse=True)[:8])

lab = lambda P: pd.Series({n: i for i, c in enumerate(P) for n in c}).reindex(list(UG.nodes))
print("NMI seeds 0 vs 1:", nmi(lab(p0), lab(p1)))

In [ ]:
# OPTIONAL (slow): effect of the resolution parameter
for res in [0.5, 1, 2]:
    parts = nx.community.louvain_communities(UG, weight="weight", resolution=res, seed=0)
    print(res, len(parts), nx.community.modularity(UG, parts, weight="weight"), sorted(map(len, parts))[-5:])

### 5.5 Do communities form around tags?

In [ ]:
generic = {"preparation", "time-to-make", "course", "main-ingredient", "dietary", "easy", "occasion", "cuisine",
           "equipment", "number-of-servings", "taste-mood"}
generic |= {t for t in recipe_tag["tags"].unique() if "minutes-or-less" in t or "hours-or-less" in t}

lab0 = lab(p0)
rc = recipes[["id", "contributor_id"]].assign(comm=recipes["contributor_id"].map(lab0)).dropna(subset=["comm"])
n_all = rc.groupby("comm").size()
big = n_all.nlargest(8).index

rt2 = recipe_tag.merge(rc[["id", "comm"]], on="id")
rt2 = rt2[~rt2["tags"].isin(generic)]
p_all = rt2.groupby("tags").size() / len(rc)
cnt = rt2[rt2["comm"].isin(big)].groupby(["comm", "tags"]).size().unstack(fill_value=0)
lift_c = cnt.div(n_all[big], axis=0).div(p_all, axis=1)
ok = cnt >= 100                                              # minimum support
cols = sorted({t for c in big for t in lift_c.loc[c].where(ok.loc[c]).nlargest(4).index})
Hm = np.log2(lift_c[cols].where(ok[cols])).fillna(-3).clip(-3, 3)
Hm.index = [f"C{int(i)} (n={n_all[i]:,})" for i in Hm.index]

cg = sns.clustermap(Hm, cmap="RdBu_r", center=0, vmin=-3, vmax=3, figsize=(12, 6))
cg.savefig("figures/metrics/community_tags.png", dpi=200)

## 6. Ingredient and tag profiles

### 6.1 Lift of each ingredient per tag

$$\text{lift}(t, i) = \frac{P(i \mid t)}{P(i)}$$

The probability that a recipe with tag $t$ contains ingredient $i$, relative to the probability that an arbitrary recipe contains it.

In [ ]:
row = pd.Series(np.arange(len(recipes)), index=recipes["id"])
t_codes, t_names = pd.factorize(recipe_tag["tags"])
i_codes, i_names = pd.factorize(recipe_ing["ingredients"])

T = sparse.csr_matrix((np.ones(len(recipe_tag)), (row[recipe_tag["id"]].values, t_codes)), shape=(len(recipes), len(t_names)))
I = sparse.csr_matrix((np.ones(len(recipe_ing)), (row[recipe_ing["id"]].values, i_codes)), shape=(len(recipes), len(i_names)))

co    = (T.T @ I).toarray()                    # tags x ingredients: recipes with both
n_tag = np.asarray(T.sum(axis=0)).ravel()      # recipes per tag
n_ing = np.asarray(I.sum(axis=0)).ravel()      # recipes per ingredient
N     = len(recipes)

p_ing_tag = co / n_tag[:, None]                # P(ingredient | tag)
p_ing_all = n_ing / N                          # P(ingredient)

lift  = pd.DataFrame(p_ing_tag / p_ing_all, index=t_names, columns=i_names)
count = pd.DataFrame(co, index=t_names, columns=i_names)

# mask out ingredients that are too rare within the tag to trust
lift = lift.where(count >= 30)

P       = pd.DataFrame(p_ing_tag, index=t_names, columns=i_names)   # P(ingredient | tag)
overall = pd.Series(p_ing_all, index=i_names)                       # P(ingredient)
CNT     = count

tag_counts = recipe_tag.groupby("tags").size().sort_values(ascending=False)

### 6.2 Helper functions

In [ ]:
def _table(tag):
    return pd.DataFrame({
        "lift": lift.loc[tag],
        "in_tag_%": (P.loc[tag] * 100).round(1),
        "overall_%": (overall * 100).round(2),
        "n": CNT.loc[tag],
    }).dropna()

def top_ingredients(tag, n=15):
    return _table(tag).nlargest(n, "lift")

def bottom_ingredients(tag, n=15):
    return _table(tag).nsmallest(n, "lift")

def signature(tag, n=8, min_count=30, min_share=0.02):
    """Ingredients that are both common in the tag and over-represented (score = share * log2 lift)."""
    d = pd.DataFrame({"lift": lift.loc[tag], "share": P.loc[tag],
                      "overall": overall, "n": CNT.loc[tag]})
    d = d[(d["n"] >= min_count) & (d["share"] >= min_share) & (d["lift"] > 1)]
    d["score"] = d["share"] * np.log2(d["lift"])
    return d.nlargest(n, "score")

In [ ]:
display(top_ingredients("desserts"))
display(top_ingredients("meat"))
display(bottom_ingredients("meat"))
display(top_ingredients("vegetarian"))
display(bottom_ingredients("vegetarian"))

### 6.3 Figure: ingredient lift per tag (dumbbell plots)

In [ ]:
tag_colors = dict(zip(["desserts", "italian", "mexican", "vegetarian", "breakfast", "seafood", "indian", "beverages"],
                      sns.color_palette("tab10", 8)))

def dumbbell(tags, path, n=8):
    fig, axes = plt.subplots(2, 2, figsize=(9, 9))
    for ax, t in zip(axes.flat, tags):
        c = tag_colors[t]
        d = signature(t, n).iloc[::-1]
        y = np.arange(len(d))
        ax.hlines(y, d["overall"] * 100, d["share"] * 100, color="#cccccc", lw=3, zorder=1)
        ax.scatter(d["overall"] * 100, y, color="#999999", s=50, zorder=2, label="all recipes")
        ax.scatter(d["share"] * 100, y, color=c, s=90, zorder=3, label=f"'{t}' recipes")
        for yi, (s, l) in enumerate(zip(d["share"] * 100, d["lift"])):
            ax.text(s + 1.8, yi, f"{l:.1f}×", va="center", fontsize=9, color=c, fontweight="bold")
        ax.set_yticks(y); ax.set_yticklabels(d.index, fontsize=10)
        ax.set_title(t, fontsize=13, fontweight="bold", color=c)
        ax.set_xlabel("% of recipes containing ingredient")
        ax.set_xlim(0, None); ax.grid(axis="x", alpha=0.3)
        for s in ["top", "right"]:
            ax.spines[s].set_visible(False)
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()

dumbbell(["desserts", "italian", "mexican", "vegetarian"], "figures/metrics/ingredient_lift_1.png")
dumbbell(["breakfast", "seafood", "indian", "beverages"],  "figures/metrics/ingredient_lift_2.png")

### 6.4 Lift versus frequency for one tag

In [ ]:
def lift_scatter(tag, label_n=15, min_count=30):
    d = pd.DataFrame({"lift": lift.loc[tag], "share": P.loc[tag], "n": CNT.loc[tag]})
    d = d[d["n"] >= min_count].dropna()
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.scatter(d["share"] * 100, d["lift"], s=12, alpha=0.4, color="#4c78a8")
    ax.axhline(1, color="k", lw=0.8, ls="--")
    ax.set_xscale("log"); ax.set_yscale("log")
    top = (d["share"] * np.log2(d["lift"].clip(lower=1))).nlargest(label_n).index
    for ing in top:
        ax.annotate(ing, (d.loc[ing, "share"] * 100, d.loc[ing, "lift"]), fontsize=8,
                    xytext=(4, 3), textcoords="offset points")
    ax.set_xlabel("% of tag's recipes containing ingredient (log)")
    ax.set_ylabel("lift (log)")
    ax.set_title(f"Ingredient signature: {tag}")
    plt.show()

lift_scatter("vegetarian")

### 6.5 Which tags are most distinctive?

In [ ]:
min_recipes = 500
k = 50

scores = {}
for t in P.index:
    if tag_counts.get(t, 0) < min_recipes:
        continue
    sig = signature(t, k)
    scores[t] = np.log2(sig["lift"]).mean()

tag_scores = pd.Series(scores).sort_values(ascending=False)
print(tag_scores.head(25))

### 6.6 Figure: cuisines and their characteristic ingredients

In [ ]:
net_tags = ["japanese", "thai", "chinese", "indian", "moroccan", "greek", "italian", "irish", "spanish", "mexican"]

B = nx.Graph()
for t in net_tags:
    B.add_node(("tag", t), kind="tag", label=t)
    for ing, r in signature(t, 20).iterrows():
        B.add_node(("ing", ing), kind="ing", label=ing)
        B.add_edge(("tag", t), ("ing", ing), w=2 * np.log2(r["lift"]))

tag_nodes = [("tag", t) for t in net_tags]
ing_nodes = [n_ for n_ in B if B.nodes[n_]["kind"] == "ing"]

R = 0.5
per_row, row_gap, arc = 3, 0.1, np.deg2rad(10)
first_ring = 0.09

ang = {t: a for t, a in zip(tag_nodes, np.linspace(0, 2 * np.pi, len(tag_nodes), endpoint=False))}
pos = {t: R * np.array([np.cos(a), np.sin(a)]) for t, a in ang.items()}

# ingredients exclusive to one tag
for t in tag_nodes:
    excl = [n_ for n_ in B.neighbors(t) if B.degree(n_) == 1]
    excl.sort(key=lambda n_: -B[t][n_]["w"])
    for i, n_ in enumerate(excl):
        row_i, j = divmod(i, per_row)
        offs = (j - (per_row - 1) / 2) / max(per_row - 1, 1) * 2 * arc
        r_ = R + first_ring + row_gap * row_i
        pos[n_] = r_ * np.array([np.cos(ang[t] + offs), np.sin(ang[t] + offs)])

# shared ingredients
for n_ in [n_ for n_ in ing_nodes if B.degree(n_) > 1]:
    pos[n_] = np.mean([pos[t] for t in B.neighbors(n_)], axis=0)

def declutter(pos, fixed, min_dist=0.04, iters=400, seed=0):
    """Push overlapping nodes apart (fixed nodes stay in place)."""
    nodes_ = [n_ for n_ in pos if n_ not in fixed]
    X = np.array([pos[n_] for n_ in nodes_])
    X += np.random.default_rng(seed).normal(scale=1e-3, size=X.shape)
    eye = np.eye(len(nodes_), dtype=bool)
    for _ in range(iters):
        dd = X[:, None] - X[None]
        dist = np.linalg.norm(dd, axis=2)
        m = (dist < min_dist) & ~eye
        if not m.any():
            break
        push = dd / np.where(eye, 1, dist)[..., None] * ((min_dist - dist) / 2)[..., None]
        X += (push * m[..., None]).sum(1)
    pos.update({n_: X[i] for i, n_ in enumerate(nodes_)})

declutter(pos, set(tag_nodes), min_dist=0.08)

tag_col = dict(zip(tag_nodes, sns.color_palette("tab10", len(tag_nodes))))
ing_col = []
for n_ in ing_nodes:
    nb = list(B.neighbors(n_))
    mix = np.mean(np.array([tag_col[t] for t in nb]) ** 2.2, axis=0) ** (1 / 2.2)   # mix tag colours (gamma-corrected)
    ing_col.append(tuple(mix))

ing_freq = np.array([overall[n_[1]] for n_ in ing_nodes])
ing_size = 150 + 250 * ing_freq / ing_freq.max()
edge_cols = [tag_col[u] if u in tag_col else tag_col[v] for u, v in B.edges]

fig, ax = plt.subplots(figsize=(18, 18))
nx.draw_networkx_edges(B, pos, width=0.8, edge_color=edge_cols, alpha=0.25, ax=ax)
nx.draw_networkx_nodes(B, pos, nodelist=ing_nodes, node_size=ing_size, node_color=ing_col, alpha=0.9, ax=ax)
nx.draw_networkx_nodes(B, pos, nodelist=tag_nodes, node_size=2500, node_color=[tag_col[t] for t in tag_nodes], ax=ax)
nx.draw_networkx_labels(B, pos, {n_: B.nodes[n_]["label"] for n_ in ing_nodes}, font_size=8, ax=ax,
                        bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.7))
nx.draw_networkx_labels(B, pos, {n_: B.nodes[n_]["label"] for n_ in tag_nodes},
                        font_size=13, font_weight="bold", font_color="white", ax=ax)
ax.set_aspect("equal"); ax.axis("off")
plt.savefig("figures/metrics/cuisines.png", dpi=200, bbox_inches="tight")
plt.show()